# E1 — ViBidLQA_v1 / MRC baseline
**Done Team | Ngày 1 | Bản đã sắp xếp lại**

Mục tiêu: chạy inference trên 20 mẫu validation, chưa fine-tune và chưa đánh giá test. Chọn checkpoint công khai `deepset/xlm-roberta-base-squad2` để pilot (không phải tái lập nguyên bản bài báo).

**Cách dùng:** mở trong Colab, chọn GPU T4, chạy theo thứ tự 01 → 12. Nếu cài thư viện mới ở 01, restart session rồi chạy lại từ 01. Không ghép thêm cell từ notebook cũ. Nếu có lỗi, dừng tại cell đó.

**Kết quả đã quan sát trong notebook gốc** (không phải output mới của bản này): T4, VRAM 14,56 GB; train 1.928 / validation 482 / test 603; train có 1.792 span duy nhất và 136 nhiều vị trí; validation có 455 và 27 tương ứng; 0 mẫu trống/không tìm thấy ở train và validation; 2.410 mẫu được chuyển đổi và kiểm tra offset thành công. Chưa nạp model hoặc chạy pilot.

Nguồn: [ViBidLQA_v1](https://huggingface.co/datasets/ntphuc149/ViBidLQA_v1), [checkpoint](https://huggingface.co/deepset/xlm-roberta-base-squad2). Theo kế hoạch v2: giữ split công bố, dùng dev cho pilot, test chỉ sau khóa cấu hình.

## 01. Cài thư viện

Khớp các phiên bản đã import thành công trên Colab của bạn. Không cài lại PyTorch/CUDA. Xung đột Gradio/Diffusers không thuộc pipeline này.

In [3]:
import sys
import subprocess
from importlib.metadata import version, PackageNotFoundError

REQUIRED = {
    "transformers": "4.57.1",
    "datasets": "4.8.5",
    "huggingface-hub": "0.36.2",
    "accelerate": "1.15.0",
    "sentencepiece": "0.2.2",
}

def installed_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None

if any(installed_version(k) != v for k, v in REQUIRED.items()):
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           *[f"{k}=={v}" for k, v in REQUIRED.items()]])
    print("Đã cài. Restart session rồi chạy lại từ mục 01.")
else:
    print("Phiên bản đã khớp; không cần cài lại.")

Phiên bản đã khớp; không cần cài lại.


## 02. Import, seed và GPU

Seed cố định việc chọn mẫu. CUDA được kiểm tra bằng phép tính thật. CPU vẫn dùng được để pilot nhỏ.

In [4]:
import sys
import platform
import json
import time
import shutil
import unicodedata
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import psutil
import transformers
import datasets
import huggingface_hub
import accelerate
import sentencepiece
from datasets import Dataset, load_dataset
from huggingface_hub import HfApi
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline, set_seed
from tqdm.auto import tqdm

SEED = 42
set_seed(SEED)
DEVICE = 0 if torch.cuda.is_available() else -1

print("Python:", platform.python_version())
for lib in (torch, transformers, datasets, huggingface_hub, accelerate, sentencepiece):
    print(lib.__name__, lib.__version__)
print("RAM (GiB):", round(psutil.virtual_memory().total / 1024**3, 2))
print("CUDA:", torch.cuda.is_available())
if DEVICE == 0:
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    print("CUDA calculation:", torch.tensor([1., 2., 3.], device="cuda") * 2)
else:
    print("CPU mode")

Python: 3.13.18
torch 2.14.1+cpu
transformers 4.57.1
datasets 4.8.5
huggingface_hub 0.36.2
accelerate 1.15.0
sentencepiece 0.2.2
RAM (GiB): 12.67
CUDA: False
CPU mode


## 03. Tải và thống kê dataset

Dùng đúng revision đã ghi trong notebook gốc. Không chia lại dữ liệu; chỉ thống kê số lượng test.

In [5]:
DATASET_NAME = "ntphuc149/ViBidLQA_v1"
dataset_revision = "dc439e1344ec6a55a36027f1868b36d93eec840d"
api = HfApi()

dataset = load_dataset(DATASET_NAME, revision=dataset_revision)
summary_df = pd.DataFrame([
    {"split": split, "num_samples": len(data),
     "columns": ", ".join(data.column_names)}
    for split, data in dataset.items()
])
display(summary_df)
print("Total:", summary_df["num_samples"].sum())
print("Revision:", dataset_revision)

assert {s: len(dataset[s]) for s in ("train", "validation", "test")} == {
    "train": 1928, "validation": 482, "test": 603
}, "Số mẫu khác snapshot đã kiểm tra; cần đối chiếu."

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

ViBidLQA_train.jsonl: 0.00B [00:00, ?B/s]

ViBidLQA_val.jsonl: 0.00B [00:00, ?B/s]

ViBidLQA_test.jsonl: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

,split,num_samples,columns
0,train,1928,"context, question, answer"
1,validation,482,"context, question, answer"
2,test,603,"context, question, answer"


Total: 3013
Revision: dc439e1344ec6a55a36027f1868b36d93eec840d


## 04. Unicode và kiểm định span

Chuẩn hóa NFC trước khi tính offset ký tự. Hàm inspect_span phải được khai báo trước khi gọi. Mọi vị trí tìm được chỉ là ứng viên khớp chuỗi.

In [6]:
def normalize_unicode(text):
    return unicodedata.normalize("NFC", text)

def inspect_span(row):
    context, question, answer = [
        normalize_unicode(row[k] or "")
        for k in ("context", "question", "answer")
    ]
    if not all(s.strip() for s in (context, question, answer)):
        return "empty", []
    positions = [m.start() for m in re.finditer(re.escape(answer), context)]
    if not positions:
        return "not_found", []
    return ("valid_unique" if len(positions) == 1 else "multiple"), positions

sample = dataset["train"][0]
print("Question:", normalize_unicode(sample["question"]))
print("Raw answer:", sample["answer"], "| length:", len(sample["answer"]))
print("NFC answer:", normalize_unicode(sample["answer"]),
      "| length:", len(normalize_unicode(sample["answer"])))
print("Span status, positions:", inspect_span(sample))

audit_records = []
for split in ("train", "validation"):
    counts = Counter(inspect_span(row)[0] for row in dataset[split])
    for status in ("empty", "multiple", "not_found", "valid_unique"):
        audit_records.append({"split": split, "status": status, "count": counts[status]})
audit_df = pd.DataFrame(audit_records)
audit_table = audit_df.pivot(index="split", columns="status", values="count")
audit_table["total"] = audit_table.sum(axis=1)
display(audit_table)

Question: Luật nào quy định về trường hợp cơ sở khám chữa bệnh tư nhân không áp dụng quy định đấu thầu trong mua thuốc, hóa chất, vật tư xét nghiệm, thiết bị y tế?
Raw answer: Luật Đấu thầu | length: 19
NFC answer: Luật Đấu thầu | length: 13
Span status, positions: ('valid_unique', [0])


status,empty,multiple,not_found,valid_unique,total
split,,,,,
train,0,136,0,1792,1928
validation,0,27,0,455,482


## 05. Chuyển sang SQuAD và kiểm định

Chỉ có một phiên bản prepared. Giữ toàn bộ span ứng viên, không loại multiple. Khi train thật sẽ chọn span ứng viên phù hợp cửa sổ token; chưa thực hiện ở notebook này.

In [7]:
def validate_mrc(row):
    texts = row["answers"]["text"]
    starts = row["answers"]["answer_start"]
    return bool(texts) and len(texts) == len(starts) and all(
        text and isinstance(start, int) and start >= 0
        and row["context"][start:start + len(text)] == text
        for text, start in zip(texts, starts)
    )

prepared = {}
for split in ("train", "validation"):
    records = []
    for i, row in enumerate(dataset[split]):
        status, positions = inspect_span(row)
        if status in ("empty", "not_found"):
            raise ValueError(f"Cần kiểm tra {split}_{i:05d}: {status}")
        context, question, answer = [
            normalize_unicode(row[k]) for k in ("context", "question", "answer")
        ]
        record = {
            "id": f"{split}_{i:05d}",
            "context": context,
            "question": question,
            "answers": {
                "text": [answer] * len(positions),
                "answer_start": positions,
            }
        }
        assert validate_mrc(record), record["id"]
        records.append(record)
    prepared[split] = Dataset.from_list(records)
    print(f"{split}: {len(prepared[split])}; invalid: 0")

print("DATASET VALIDATION SUCCESSFUL!")
print("First answers:", prepared["train"][0]["answers"])
multiple_example = next((r for r in prepared["train"] if len(r["answers"]["text"]) > 1), None)
if multiple_example:
    print("Multiple example:", multiple_example["id"], multiple_example["answers"])

DATA_DIR = Path("research/datasets/vibidlqa")
DATA_DIR.mkdir(parents=True, exist_ok=True)
for split in ("train", "validation"):
    prepared[split].to_json(str(DATA_DIR / f"{split}.jsonl"), force_ascii=False)

train: 1928; invalid: 0
validation: 482; invalid: 0
DATASET VALIDATION SUCCESSFUL!
First answers: {'answer_start': [0], 'text': ['Luật Đấu thầu']}
Multiple example: train_00003 {'answer_start': [705, 1129], 'text': ['gửi đơn kiến nghị đến người có thẩm quyền thông qua bộ phận thường trực', 'gửi đơn kiến nghị đến người có thẩm quyền thông qua bộ phận thường trực']}


Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

## 06. Nạp checkpoint và tokenizer

Đây là mốc tiếp theo chưa chạy trong notebook gốc. Mô hình công khai đã fine-tune SQuAD 2.0; chưa fine-tune bởi nhóm. Chưa thay bằng checkpoint khác khi chưa ghi nhận quyết định.

In [8]:
MODEL_NAME = "deepset/xlm-roberta-base-squad2"
model_revision = api.model_info(MODEL_NAME).sha

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, revision=model_revision, use_fast=True
)
model, loading_info = AutoModelForQuestionAnswering.from_pretrained(
    MODEL_NAME, revision=model_revision, output_loading_info=True
)
print("Model:", MODEL_NAME)
print("Revision:", model_revision)
print("Architecture:", model.config.model_type)
print("Tokenizer:", type(tokenizer).__name__)
print("Fast tokenizer:", tokenizer.is_fast)
print("Missing keys:", loading_info["missing_keys"])
print("Mismatched keys:", loading_info["mismatched_keys"])
print("Parameters:", sum(p.numel() for p in model.parameters()))

assert tokenizer.is_fast
assert not loading_info["missing_keys"], "Kiểm tra trọng số bị thiếu."
assert not loading_info["mismatched_keys"], "Kiểm tra trọng số không khớp."
print("MODEL LOADED SUCCESSFULLY!")

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Model: deepset/xlm-roberta-base-squad2
Revision: a5fab9908c8d856e8c583fd41ba6d92444e46477
Architecture: xlm-roberta
Tokenizer: XLMRobertaTokenizerFast
Fast tokenizer: True
Missing keys: []
Mismatched keys: []
Parameters: 277454594
MODEL LOADED SUCCESSFULLY!


## 07. Thử tokenizer và offset mapping

Offset token được dùng để ánh xạ về chuỗi gốc. Ví dụ chỉ để học, không dùng chấm điểm.

In [9]:
text = "Phường Sài Gòn được hình thành sau sáp nhập."
encoded = tokenizer(text, return_offsets_mapping=True)
print("Input IDs:", encoded["input_ids"])
print("Attention mask:", encoded["attention_mask"])
for token, (start, end) in zip(
    tokenizer.convert_ids_to_tokens(encoded["input_ids"]),
    encoded["offset_mapping"]
):
    print(f"{token:20} ({start:2}, {end:2}) {text[start:end]}")

Input IDs: [0, 116041, 84610, 89090, 912, 4609, 2781, 858, 91, 50731, 11820, 5, 2]
Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
<s>                  ( 0,  0) 
▁Phường              ( 0,  6) Phường
▁Sài                 ( 6, 10)  Sài
▁Gòn                 (10, 14)  Gòn
▁được                (14, 19)  được
▁hình                (19, 24)  hình
▁thành               (24, 30)  thành
▁sau                 (30, 34)  sau
▁s                   (34, 36)  s
áp                   (36, 38) áp
▁nhập                (38, 43)  nhập
.                    (43, 44) .
</s>                 ( 0,  0) 


## 08. Inference một câu địa giới và một câu validation

Context cho trước. Không tự truy xuất DB. Tham số thử nghiệm được giữ nguyên cho pilot. handle_impossible_answer=False vì đánh giá câu có đáp án.

In [10]:
qa_pipeline = pipeline(
    task="question-answering",
    model=model,
    tokenizer=tokenizer,
    device=DEVICE
)

QA_SETTINGS = {
    "max_seq_len": 384,
    "doc_stride": 128,
    "max_answer_len": 128,
    "handle_impossible_answer": False
}

print("QA Pipeline is ready!")

example_context = "Toàn bộ phường Bến Nghé cũ được sáp nhập vào phường Sài Gòn."
example_question = "Phường Bến Nghé cũ nay thuộc phường nào?"
print("Ví dụ địa giới:", qa_pipeline(question=example_question, context=example_context, **QA_SETTINGS))

sample = prepared["validation"][0]
prediction = qa_pipeline(question=sample["question"], context=sample["context"], **QA_SETTINGS)
print("Question:", sample["question"])
print("Gold:", sample["answers"]["text"][0])
print("Prediction:", prediction)

Device set to use cpu


QA Pipeline is ready!
Ví dụ địa giới: {'score': 0.44437638483941555, 'start': 44, 'end': 60, 'answer': ' phường Sài Gòn.'}
Question: Thông tin về lựa chọn nhà thầu theo quy định tại khoản 1 Điều 21 Nghị định hướng dẫn Luật Đấu thầu số 22 được đăng tải công khai ở đâu?
Gold: Hệ thống mạng đấu thầu quốc gia và Báo Đấu thầu
Prediction: {'score': 0.617105957120657, 'start': 163, 'end': 212, 'answer': ' Hệ thống mạng đấu thầu quốc gia và Báo Đấu thầu.'}


## 09. Định nghĩa EM và F1

Chuẩn hóa NFC, chữ thường, khoảng trắng; giữ dấu và dấu câu. F1 tính theo token cách nhau bởi khoảng trắng. Quy tắc nhóm đề xuất, không mặc định trùng bài báo.

In [11]:
from collections import Counter

def normalize_answer(text):
    text = unicodedata.normalize("NFC", text)
    text = text.lower().strip()
    text = " ".join(text.split())
    return text


def exact_match(prediction, gold):
    return int(
        normalize_answer(prediction)
        == normalize_answer(gold)
    )


def f1_score(prediction, gold):
    pred_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(gold).split()

    if not pred_tokens or not gold_tokens:
        return float(pred_tokens == gold_tokens)

    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)
assert exact_match("phường Sài Gòn", "phường Sài Gòn") == 1
assert abs(f1_score("Sài Gòn", "phường Sài Gòn") - 0.8) < 1e-9
print("Metric self-check OK")

Metric self-check OK


## 10. Pilot 20 mẫu validation

Chọn ngẫu nhiên với seed 42 từ toàn bộ 482 mẫu đã chuẩn hóa, bao gồm cả multiple. Không mở test. Độ trễ là thời gian gọi pipeline tuần tự trên Colab, không so trực tiếp với độ trễ backend.

In [12]:
PILOT_SIZE = 20
pilot_dataset = prepared["validation"].shuffle(seed=SEED).select(
    range(min(PILOT_SIZE, len(prepared["validation"])))
)

results = []
for row in tqdm(pilot_dataset):
    gold = row["answers"]["text"][0]
    if DEVICE == 0:
        torch.cuda.synchronize()
    start_time = time.perf_counter()
    output = qa_pipeline(question=row["question"], context=row["context"], **QA_SETTINGS)
    if DEVICE == 0:
        torch.cuda.synchronize()
    elapsed = (time.perf_counter() - start_time) * 1000
    pred = output["answer"]
    results.append({
        "id": row["id"],
        "split": "validation",
        "model": MODEL_NAME,
        "context": row["context"],
        "question": row["question"],
        "gold_answer": gold,
        "gold_starts": row["answers"]["answer_start"],
        "prediction": pred,
        "score": float(output["score"]),
        "pred_start": output["start"],
        "pred_end": output["end"],
        "exact_match": exact_match(pred, gold),
        "f1": f1_score(pred, gold),
        "latency_ms": elapsed
    })
results_df = pd.DataFrame(results)
display(results_df[["id", "question", "gold_answer", "prediction", "exact_match", "f1"]])

  0%|          | 0/20 [00:00<?, ?it/s]

,id,question,gold_answer,prediction,exact_match,f1
0,validation_00479,Luật Đấu thầu số 22/2023/QH15 Điều 37 quy định...,lập kế hoạch lựa chọn nhà thầu,lập kế hoạch lựa chọn nhà thầu khoản 5,0,0.875000
1,validation_00294,Nhà thầu có thời hạn tối đa bao lâu để gửi phả...,20 ngày,20 ngày,1,1.000000
2,validation_00131,"Trong quá trình đánh giá hồ sơ đề xuất, đơn gi...",không được vượt giá trị trước thuế của hàng hó...,"đánh giá về tư cách hợp lệ, năng lực kinh ngh...",0,0.139535
3,validation_00375,Thông tin nào được công khai trên Hệ thống mạn...,"kết quả thực hiện hợp đồng của nhà thầu, chất ...",Thông tin về kết quả thực hiện hợp đồng của n...,0,0.642857
4,validation_00471,Nhà thầu có nhiệm vụ gì đối với gói thầu mà nh...,"có trách nhiệm theo dõi, cập nhật các thông ti...","theo dõi, cập nhật các thông tin trên Hệ thốn...",0,0.909091
5,validation_00040,Hồ sơ nào được coi là căn cứ để thương thảo hợ...,Hồ sơ mời thầu,Báo cáo kết quả đánh giá hồ sơ dự thầu;,0,0.285714
6,validation_00423,Ttrường hợp hồ sơ mời thầu có các quy định nào...,có các quy định gây hạn chế cạnh tranh theo qu...,gây hạn chế cạnh tranh,0,0.384615
7,validation_00152,"Theo quy định của Luật Đấu thầu, chủ thể nào q...","người có thẩm quyền, chủ đầu tư",chủ đầu tư,0,0.600000
8,validation_00268,Việc chỉ định thầu rút gọn đối với trường hợp ...,lựa chọn nhà thầu,lựa chọn nhà thầu.,0,0.750000
9,validation_00180,Luật Đấu thầu số 22/2023/QH15 Điều 42 quy định...,Đấu thầu trước,Đấu thầu trước khoản 2,0,0.750000


## 11. Tổng hợp và phân tích lỗi

EM/F1 là phần trăm trung bình trên 20 câu pilot. Chưa phải kết quả test chính thức. EM=0 không nhất thiết sai về ngữ nghĩa.

In [13]:
metrics = {
    "model": MODEL_NAME,
    "dataset": DATASET_NAME,
    "split": "validation",
    "num_samples": len(results_df),
    "exact_match": float(results_df["exact_match"].mean() * 100),
    "f1": float(results_df["f1"].mean() * 100),
    "avg_latency_ms": float(results_df["latency_ms"].mean())
}

display(pd.DataFrame([metrics]))

wrong_answers = results_df[results_df["exact_match"] == 0].copy()
display(wrong_answers[["id", "question", "gold_answer", "prediction", "f1"]])

,model,dataset,split,num_samples,exact_match,f1,avg_latency_ms
0,deepset/xlm-roberta-base-squad2,ntphuc149/ViBidLQA_v1,validation,20,5.0,61.135556,598.163344


,id,question,gold_answer,prediction,f1
0,validation_00479,Luật Đấu thầu số 22/2023/QH15 Điều 37 quy định...,lập kế hoạch lựa chọn nhà thầu,lập kế hoạch lựa chọn nhà thầu khoản 5,0.875000
2,validation_00131,"Trong quá trình đánh giá hồ sơ đề xuất, đơn gi...",không được vượt giá trị trước thuế của hàng hó...,"đánh giá về tư cách hợp lệ, năng lực kinh ngh...",0.139535
3,validation_00375,Thông tin nào được công khai trên Hệ thống mạn...,"kết quả thực hiện hợp đồng của nhà thầu, chất ...",Thông tin về kết quả thực hiện hợp đồng của n...,0.642857
4,validation_00471,Nhà thầu có nhiệm vụ gì đối với gói thầu mà nh...,"có trách nhiệm theo dõi, cập nhật các thông ti...","theo dõi, cập nhật các thông tin trên Hệ thốn...",0.909091
5,validation_00040,Hồ sơ nào được coi là căn cứ để thương thảo hợ...,Hồ sơ mời thầu,Báo cáo kết quả đánh giá hồ sơ dự thầu;,0.285714
6,validation_00423,Ttrường hợp hồ sơ mời thầu có các quy định nào...,có các quy định gây hạn chế cạnh tranh theo qu...,gây hạn chế cạnh tranh,0.384615
7,validation_00152,"Theo quy định của Luật Đấu thầu, chủ thể nào q...","người có thẩm quyền, chủ đầu tư",chủ đầu tư,0.600000
8,validation_00268,Việc chỉ định thầu rút gọn đối với trường hợp ...,lựa chọn nhà thầu,lựa chọn nhà thầu.,0.750000
9,validation_00180,Luật Đấu thầu số 22/2023/QH15 Điều 42 quy định...,Đấu thầu trước,Đấu thầu trước khoản 2,0.750000
10,validation_00328,"Theo Nghị định hướng dẫn Luật Đấu thầu, khi nà...",Hệ thống mạng đấu thầu quốc gia hiển thị thông...,trong trường hợp nhà thầu không chấp nhận đơn...,0.600000


## 12. Lưu kết quả và tải ZIP

Xuất dự đoán, thống kê, lỗi, cấu hình và dữ liệu đã chuyển đổi. Tải ZIP về máy; notebook cũng cần lưu bằng menu Tệp → Tải xuống .ipynb.

In [14]:
OUTPUT_DIR = Path("research/results/E1_pilot")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
config = {
    "experiment": "E1_pilot",
    "dataset": DATASET_NAME,
    "dataset_revision": dataset_revision,
    "model": MODEL_NAME,
    "model_revision": model_revision,
    "tokenizer": type(tokenizer).__name__,
    "fast_tokenizer": tokenizer.is_fast,
    "fine_tuned_by_team": False,
    "seed": SEED,
    "split": "validation",
    "pilot_size": len(pilot_dataset),
    "pilot_ids": list(pilot_dataset["id"]),
    "pilot_filter": "all validated spans (unique and multiple)",
    "qa_settings": QA_SETTINGS,
    "python_version": platform.python_version(),
    "libraries": {lib.__name__: str(lib.__version__) for lib in (
        torch, transformers, datasets, huggingface_hub, accelerate, sentencepiece
    )},
    "device": "cuda" if DEVICE == 0 else "cpu",
    "gpu": torch.cuda.get_device_name(0) if DEVICE == 0 else None,
    "metric_normalization": "NFC, lowercase, whitespace; retain punctuation and Vietnamese accents"
}

for name, frame in (
    ("dataset_summary", summary_df), ("span_audit", audit_df),
    ("predictions", results_df), ("errors", wrong_answers)
):
    frame.to_csv(OUTPUT_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")

for name, obj in (("metrics", metrics), ("config", config)):
    (OUTPUT_DIR / f"{name}.json").write_text(
        json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8"
    )

for split in ("train", "validation"):
    shutil.copy2(DATA_DIR / f"{split}.jsonl", OUTPUT_DIR / f"{split}.jsonl")

archive = shutil.make_archive("E1_pilot_results", "zip", root_dir=str(OUTPUT_DIR))
print("Saved:", archive)
from google.colab import files
files.download(archive)

Saved: /content/E1_pilot_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>